In [1]:
import mlflow
from mlflow.tracking import MlflowClient

tracking_uri = "http://127.0.0.1:5000"
experiment_name = "nyc-taxi-duration"
registered_model_name = "nyc-taxi-duration-model"

mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient(tracking_uri=tracking_uri)

In [2]:
experiments = client.search_experiments()

for experiment in experiments:
    print(experiment.experiment_id, experiment.name)


1 nyc-taxi-duration
0 Default


In [3]:
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    raise ValueError(f"Expérience introuvable : {experiment_name}")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.validation_rmse ASC"],
    max_results=5,
)

if runs.empty:
    raise ValueError("Aucune exécution terminée n'a été trouvée")

best_run = runs.iloc[0]
runs[["run_id", "params.model", "metrics.validation_rmse"]]


,run_id,params.model,metrics.validation_rmse
0,e0019a8efe52488faf9b7557d8e16e86,NaN,6.455738
1,aca2564a9f0e42409e9316bd098e04e2,NaN,6.455738
2,7979cd2b7a7a44888a7d69e07768181b,XGBoost,6.455738


In [4]:
run_id = best_run["run_id"]
model_uri = f"runs:/{run_id}/model"

model_version = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name,
)

model_version.version


Registered model 'nyc-taxi-duration-model' already exists. Creating a new version of this model...
2026/11/27 15:24:21 WARNING mlflow.tracking._model_registry.fluent: Run with id e0019a8efe52488faf9b7557d8e16e86 has no artifacts at artifact path 'model', registering model based on models:/m-c778d042ad824a2993c73ec767c8bffd instead
2026/11/27 15:24:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-duration-model, version 2
Created version '2' of model 'nyc-taxi-duration-model'.


'2'

In [9]:
versions = client.search_model_versions(
    filter_string=f"name = '{registered_model_name}'"
)

for item in versions:
    print(
        "version:", item.version,
        "run:", item.run_id,
        "status:", item.status,
    )


version: 2 run: e0019a8efe52488faf9b7557d8e16e86 status: READY
version: 1 run: e0019a8efe52488faf9b7557d8e16e86 status: READY


In [18]:
version = model_version.version

client.update_model_version(
    name=registered_model_name,
    version=version,
    description="Modèle candidat pour la prédiction de durée des trajets Yellow Taxi.",
)

client.set_model_version_tag(
    name=registered_model_name,
    version=version,
    key="validation_status",
    value="reviewed",
)



In [19]:
client.set_registered_model_alias(
    name=registered_model_name,
    alias="champion",
    version=version,
)


In [21]:
champion_version = client.get_model_version_by_alias(
    name=registered_model_name,
    alias="champion",
)

champion_version.version, champion_version.run_id
champion_version

<ModelVersion: aliases=['champion'], creation_timestamp=1795793061825, current_stage='None', deployment_job_state=None, description='Modèle candidat pour la prédiction de durée des trajets Yellow Taxi.', last_updated_timestamp=1795793218298, metrics=None, model_id=None, name='nyc-taxi-duration-model', params=None, run_id='e0019a8efe52488faf9b7557d8e16e86', run_link='', source='models:/m-c778d042ad824a2993c73ec767c8bffd', status='READY', status_message=None, tags={'validation_status': 'reviewed'}, user_id='', version='2', workspace='default'>

In [22]:

champion_uri = f"models:/{registered_model_name}@champion"
champion_model = mlflow.sklearn.load_model(champion_uri)

champion_predictions = champion_model.predict(X_val[:5])
champion_predictions


MlflowException: Model does not have the "sklearn" flavor

In [23]:
local_path = mlflow.artifacts.download_artifacts(
    f"runs:/{run_id}/pickle/duration_model.bin"  # chemin affiché dans Artifacts
)

MlflowException: Failed to download artifacts from path 'duration_model.bin', please ensure that the path is correct.

In [ ]:
registered_models = client.search_registered_models()

for model in registered_models:
    print(model.name)


In [ ]:
download_path = client.download_artifacts(
    run_id=run_id,
    path="model",
    dst_path="downloaded-artifacts",
)

download_path
